#  <center> Taller  de Aprendizaje Automático </center>
##  <center> Taller 5: Estimación de la demanda de bicicletas compartidas utilizando *Neural Networks*.  </center>

# Introducción

En esta actividad se retomará el problema de la competencia [*Bike Sharing Demand*](https://www.kaggle.com/c/bike-sharing-demand) visto en el Taller 3.
Esta vez las estimaciónes deben obtenerse utilizando la herramienta: *Multilayer Perceptron* (MLP). Es importante mantener la función *Root Mean Squared Logarithmic Error* (RMSLE) como medida de desempeño de manera de poder comparar los resultados con los obtenidos en el Taller 3.

Tanto las preguntas teóricas como la parte práctica de esta actividad están ligadas al contenido del capítulo 10 (*Introduction to
Artificial Neural Networks with
Keras*) del libro del curso.


## Objetivos


*   Trabajar con modelos MLP utilizando la librería [*Keras*](https://keras.io/api/).
*   Probar algunas de las herramientas disponibles para la busqueda de hiperparámetros.

## Formas de trabajo

#### Opción 1: Trabajar localmente

##### Descarga de datos disponibles en Kaggle

Luego, para descargar el dataset de Demanda de Bicicletas:

In [ ]:
!kaggle competitions download -c bike-sharing-demand

Descomprima el archivo descargado:

In [ ]:
import shutil
shutil.unpack_archive('./bike-sharing-demand.zip', './')

#### Opción 2:  Trabajar en *Colab*.

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/TAA-fing/TAA-2025/blob/main/talleres/taller5_demanda_de_bicicletas_con_NNs.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Ejecutar en Google Colab</a>
  </td>
</table>

Se puede trabajar en Google Colab. Para ello es necesario contar con una cuenta de **google drive** y ejecutar un notebook almacenado en dicha cuenta. De lo contrario, no se conservarán los cambios realizados en la sesión. En caso de ya contar con una cuenta, se puede abrir el notebook y luego ir a `Archivo-->Guardar una copia en drive`.

La siguiente celda monta el disco personal del drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

A continuación, vaya a su cuenta de [Kaggle](https://www.kaggle.com/) (o cree una si aún no lo ha hecho), haga clic en el icono de perfil en la esquina superior derecha de la pantalla y seleccione "Your Account" en la lista desplegable. Luego, seleccione la viñeta "Account" y haga clic en "Create new API token". Entonces un archivo llamado kaggle.json se descargará automáticamente a su carpeta de descargas. Este archivo contiene sus credenciales de inicio de sesión para permitirle acceder a la API.

La siguiente celda realiza la configuración necesaria para obtener datos desde la plataforma Kaggle. Le solicitará que suba el archivo kaggle.json descargado anteriormente.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from google.colab import files

# El siguiente archivo solicitado es para habilitar la API de Kaggle en el entorno que está trabajando.
# Este archivo se descarga entrando a su perfíl de Kaggle, en la sección API, presionando donde dice: Create New API Token

uploaded = files.upload()

for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

#Then move kaggle.json into the folder where the API expects to find it.
!mkdir -p ~/.kaggle/ && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

Una vez guardado el *token* se pueden descargar los datos, en este caso se bajarán los datos del dataset de Demanda de Bicicletas:

In [ ]:
!kaggle competitions download -c bike-sharing-demand

In [ ]:
!unzip bike-sharing-demand.zip

## Paquetes a utilizar

En esta actividad se utilizarán algunas bibliotecas auxiliares que deberán ser instaladas. Ejecutar la siguiente celda hasta que se ejecute sin errores. En caso de error, se puede instalar el paquete faltante desde el notebook con el comando:

`!pip install paquete_faltante`

In [ ]:
!pip install comet_ml
!pip install keras_tuner

In [ ]:
#import comet_ml in the top of your file
from comet_ml import Experiment
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import keras_tuner as kt
import seaborn as sns
sns.set_theme(style="whitegrid")

df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_submission = pd.read_csv('sampleSubmission.csv')

## Parte 1 - Procesamiento de los datos

Dado que ya se ha familiarizado con los datos, se implementa el mismo preprocesamiento que utilizó en el Taller 3:


*   Se obtiene la hora, día de la semana, mes y año a partir de la fecha.
*   La cantidad se saca de los datos y se pone como etiqueta (y_train).
*   Se eliminan las columnas "datetime", "casual", "registered" y "count".


In [ ]:
df_train['datetime'] = pd.to_datetime(df_train['datetime'])

df_train['hour'] = df_train['datetime'].dt.hour
df_train['weekday'] = df_train['datetime'].dt.weekday
df_train['month'] = df_train['datetime'].dt.month
df_train['year'] = df_train['datetime'].dt.year

y_train_full = df_train['count']
df_train = df_train.drop(['datetime', 'casual', 'registered', 'count'], axis=1) # hay que eliminarlas ya que tiene relación directa con la columna objetivo y no aparecen en el conjunto de *test*.

- Dado que se trabajará con redes neuronales, ¿Cree conveniente realizar alguna modificación en el preprocesamiento?. De ser así, impleméntelo.

## Parte 2 - Multilayer Perceptron (MLP)

Siguiendo el ejemplo de la sección *Building a Regression MLP Using the Sequential API*:


*   Implementar un estimador manteniendo los hiperparámetros del ejemplo.
*   ¿Cuál es la cantidad total de parámetros entrenables de la red?
*   Seleccionar aleatoriamente un *10%* de los datos para validación, y graficar la función de *loss* (*Mean Squared Logarithmic Error*) de entrenamiento y validación.


*Nota 1: El ejemplo mencionado aparece en la tercera edición del libro: en versiones anteriores es distinto.*

*Nota 2: Observe que en el ejemplo se agrega una capa de normalización*

## Parte 3 - Ajuste fino (Keras Tuner)

Siguiendo el ejemplo de la sección *Fine-Tuning Neural Network Hyperparameters*:


*   Utilizar la herramienta *RandomSearch* de *KerasTuner* para la busqueda de hiperparámetros del modelo implementado en *keras*.
*   Probar el *tip* que se sugiere en la sección *Number of Neurons per Hidden Layer* y comentar los resultados.

*Nota: Keras Tuner permite realizar Optimización bayesiana.*

## Parte 4 - Ajuste fino (Optuna)

*   Utilizar *Optuna* para la busqueda de hiperparámetros del modelo en *Keras*. Se le sugiere seguir uno de los siguientes ejemplos: [*keras_simple*](https://github.com/optuna/optuna-examples/blob/main/keras/keras_simple.py),  [OptunaSearchCV](https://github.com/optuna/optuna-examples/blob/main/sklearn/sklearn_optuna_search_cv_simple.py).

*Nota 1: Optuna puede utilizarse para optimizar otras técnicas por fuera de las redes neuronales, y otros frameworks además de Keras.*


In [ ]:
!pip install optuna

Para usar OptunaSearchCV con Keras, es necesario instalar dos paquetes: ``scikeras``y ``scikit-learn``. Sin embargo, por temas de compatibilidad, las últimas versiones a la fecha (abril 2026) no son compatibles. Es por esta razón que si se quiere usar OptunaSearchCV deben instalarse versiones previas:


In [ ]:
!pip install scikeras==0.13.0
!pip install scikit-learn==1.4.2
!pip install optuna-integration[sklearn]

## Parte 5 - Pipeline



*   Incorporar el estimador con mejor desempeño a un *pipeline* similar al implementado en el Taller 3. Puede ser útil la biblioteca [scikeras](https://adriangb.com/scikeras/stable/migration.html).
*   Subir los resultados de los datos *test* a la página de la competencia.

